In [3]:
# Monte Carlo (100 iteraciones) — Demanda diaria de agua embotellada
# Modelo: Demanda  Normal(media=500, sd=50)
# Costos/Precio: costo producción = 15, precio venta = 25, almacenamiento sobrantes = 2

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# 1) Parámetros 

MU_DEMANDA = 500
SIGMA_DEMANDA = 50

COSTO_PROD = 15
PRECIO_VENTA = 25
COSTO_ALMACEN = 2

N_ITER = 100
SEED = 42  

# Producción a evaluar 
Q_INICIAL = 500

# Rango de políticas de producción 
Q_MIN, Q_MAX, STEP = 300, 700, 5  
Q_GRID = np.arange(Q_MIN, Q_MAX + 1, STEP)


# 2) Generar demandas simuladas

rng = np.random.default_rng(SEED)
demanda = rng.normal(loc=MU_DEMANDA, scale=SIGMA_DEMANDA, size=N_ITER)

# Convertir a demanda entera y no-negativa 
demanda = np.rint(demanda).astype(int)
demanda = np.clip(demanda, 0, None)


# # 3) Función de utilidad/profit

def utilidad_por_dia(Q: int, demanda_array: np.ndarray) -> np.ndarray:
    
    vendidas = np.minimum(demanda_array, Q)
    sobrantes = np.maximum(Q - demanda_array, 0)

    ingresos = PRECIO_VENTA * vendidas
    costo_produccion = COSTO_PROD * Q
    costo_almacen = COSTO_ALMACEN * sobrantes

    utilidad = ingresos - costo_produccion - costo_almacen
    return utilidad


# Pregunta 1 Simulación para una política Q fija

utilidades_Q_inicial = utilidad_por_dia(Q_INICIAL, demanda)

resumen_Q_inicial = {
    "Q": Q_INICIAL,
    "iteraciones": N_ITER,
    "utilidad_promedio": utilidades_Q_inicial.mean(),
    "utilidad_std": utilidades_Q_inicial.std(ddof=1),
    "utilidad_min": utilidades_Q_inicial.min(),
    "utilidad_max": utilidades_Q_inicial.max(),
}

print("=== RESULTADO MONTE CARLO (Q fija) ===")
for k, v in resumen_Q_inicial.items():
    print(f"{k}: {v}")

# ver tabla de las 10 primeras iteraciones
df_iter = pd.DataFrame({
    "iter": np.arange(1, N_ITER + 1),
    "demanda": demanda,
    "utilidad": utilidades_Q_inicial
})
print("\n=== Primeras 10 iteraciones (Q fija) ===")
print(df_iter.head(10).to_string(index=False))


#Pregunta 2 Buscar política de producción óptima en un grid

medias = []
stds = []

for Q in Q_GRID:
    u = utilidad_por_dia(int(Q), demanda)
    medias.append(u.mean())
    stds.append(u.std(ddof=1))

medias = np.array(medias)
stds = np.array(stds)

idx_best = np.argmax(medias)
Q_optimo = int(Q_GRID[idx_best])

resumen_grid = pd.DataFrame({
    "Q": Q_GRID.astype(int),
    "utilidad_promedio": medias,
    "utilidad_std": stds
}).sort_values("utilidad_promedio", ascending=False)

print("\n=== TOP 10 Políticas por utilidad esperada ===")
print(resumen_grid.head(10).to_string(index=False))

print("\n=== POLÍTICA ÓPTIMA===")
print(f"Q_optimo: {Q_optimo}")
print(f"utilidad_promedio_optima: {medias[idx_best]}")
print(f"utilidad_std_optima: {stds[idx_best]}")





=== RESULTADO MONTE CARLO (Q fija) ===
Q: 500
iteraciones: 100
utilidad_promedio: 4534.79
utilidad_std: 651.4211394398335
utilidad_min: 2354
utilidad_max: 5000

=== Primeras 10 iteraciones (Q fija) ===
 iter  demanda  utilidad
    1      515      5000
    2      448      3596
    3      538      5000
    4      547      5000
    5      402      2354
    6      435      3245
    7      506      5000
    8      484      4568
    9      499      4973
   10      457      3839

=== TOP 10 Políticas por utilidad esperada ===
  Q  utilidad_promedio  utilidad_std
485            4566.77    508.342421
480            4565.64    462.219405
490            4560.34    554.812922
475            4554.25    414.699754
495            4550.13    602.977958
470            4538.27    368.160808
500            4534.79    651.421139
465            4519.86    324.372059
505            4515.13    700.169239
460            4498.21    283.252619

=== POLÍTICA ÓPTIMA===
Q_optimo: 485
utilidad_promedio_optima: 4566

In [ ]:
# Tema 2 — Programación Lineal + Análisis de Sensibilidad (material ±50 kg)
import numpy as np
import pandas as pd

# 1) Formulación del modelo

# Variables de decisión:
#   xA = cantidad de paneles tipo A a producir
#   xB = cantidad de paneles tipo B a producir
#
# Max Z = 100*xA + 180*xB
#
# Sujeto a:
#   (Mano de obra) 2*xA + 3*xB <= 600
#   (Material)     1*xA + 2*xB <= 400
#   xA, xB >= 0
#

# 2) Resolver con pulp (Similar a solver de excel)


from pulp import (
    LpProblem, LpMaximize, LpVariable, lpSum, LpStatus, value, PULP_CBC_CMD
)

def resolver_modelo(max_mano_obra=600, max_material=400, enteros=False, msg=False):
 
    # Definir el problema
    model = LpProblem("EcoSmart_PL", LpMaximize)

    # Variables
    cat = "Integer" if enteros else "Continuous"
    xA = LpVariable("xA", lowBound=0, cat=cat)
    xB = LpVariable("xB", lowBound=0, cat=cat)

    # Función objetivo
    model += 100 * xA + 180 * xB, "Utilidad_Total"

    # Restricciones
    model += 2 * xA + 3 * xB <= max_mano_obra, "Mano_Obra"
    model += 1 * xA + 2 * xB <= max_material, "Material"

    # Resolver
    solver = PULP_CBC_CMD(msg=msg)
    model.solve(solver)

    # Resultados
    status = LpStatus[model.status]
    res = {
        "status": status,
        "max_mano_obra": max_mano_obra,
        "max_material": max_material,
        "xA": value(xA),
        "xB": value(xB),
        "utilidad": value(model.objective),
    }

    return res

# Caso base (material = 400 kg)
base = resolver_modelo(max_mano_obra=600, max_material=400, enteros=False, msg=False)

print("=== SOLUCIÓN ÓPTIMA (CASO BASE) ===")
for k, v in base.items():
    print(f"{k}: {v}")


# 3) Análisis de sensibilidad: material ±50 kg

sens = []
for mat in [400 - 50, 400, 400 + 50]:
    sens.append(resolver_modelo(max_mano_obra=600, max_material=mat, enteros=False, msg=False))

df_sens = pd.DataFrame(sens)
print("\n=== ANÁLISIS DE SENSIBILIDAD (MATERIAL: 350, 400, 450) ===")
print(df_sens.to_string(index=False))

#comparar cambios entre escenarios
df_comp = df_sens.copy()
df_comp["delta_xA_vs_base"] = df_comp["xA"] - base["xA"]
df_comp["delta_xB_vs_base"] = df_comp["xB"] - base["xB"]
df_comp["delta_utilidad_vs_base"] = df_comp["utilidad"] - base["utilidad"]

print("\n=== CAMBIOS VS CASO BASE ===")
print(df_comp[["max_material", "delta_xA_vs_base", "delta_xB_vs_base", "delta_utilidad_vs_base"]].to_string(index=False))


=== SOLUCIÓN ÓPTIMA (CASO BASE) ===
status: Optimal
max_mano_obra: 600
max_material: 400
xA: 1.5347723e-12
xB: 200.0
utilidad: 36000.00000000015

=== ANÁLISIS DE SENSIBILIDAD (MATERIAL: 350, 400, 450) ===
 status  max_mano_obra  max_material           xA    xB  utilidad
Optimal            600           350 1.500000e+02 100.0   33000.0
Optimal            600           400 1.534772e-12 200.0   36000.0
Optimal            600           450 0.000000e+00 200.0   36000.0

=== CAMBIOS VS CASO BASE ===
 max_material  delta_xA_vs_base  delta_xB_vs_base  delta_utilidad_vs_base
          350      1.500000e+02            -100.0           -3.000000e+03
          400      0.000000e+00               0.0            0.000000e+00
          450     -1.534772e-12               0.0           -1.527951e-10


In [ ]:
# Tema 3 — Modelo de Inventario (EOQ + ROP + Sensibilidad en costo de mantenimiento)
import math
import pandas as pd


# 1) Parámetros     
D = 5000          # Demanda anual (unidades/año)
S = 200           # Costo por pedido (RD$ por orden)
H = 10            # Costo de mantener inventario (RD$ por unidad por año)
C = 300           # Costo de compra (RD$ por unidad) -> no cambia el EOQ básico, pero puede usarse en costo total
L_dias = 5        # Lead time (días)
dias_operacion = 250  # Días operativos por año


# 2) Funciones (EOQ y ROP)

def eoq(D, S, H):
    
    return math.sqrt((2 * D * S) / H)

def demanda_diaria(D, dias_operacion):
   
    return D / dias_operacion

def rop(D, dias_operacion, L_dias):
    
    d = demanda_diaria(D, dias_operacion)
    return d * L_dias

def politicas_inventario(D, S, H, C, L_dias, dias_operacion):
    """Calcula EOQ y ROP para un conjunto de parámetros."""
    Q = eoq(D, S, H)
    R = rop(D, dias_operacion, L_dias)

    # Métricas adicionales 
    pedidos_por_anio = D / Q
    tiempo_entre_pedidos_dias = dias_operacion / pedidos_por_anio  

    # Costo total anual relevante (sin descuentos): pedido + mantenimiento + compra
    costo_pedido_anual = (D / Q) * S
    costo_mant_anual = (Q / 2) * H
    costo_compra_anual = D * C
    costo_total_anual = costo_pedido_anual + costo_mant_anual + costo_compra_anual

    return {
        "D": D,
        "S": S,
        "H": H,
        "C": C,
        "L_dias": L_dias,
        "dias_operacion": dias_operacion,
        "EOQ_Q": Q,
        "ROP_R": R,
        "pedidos_por_anio": pedidos_por_anio,
        "tiempo_entre_pedidos_dias": tiempo_entre_pedidos_dias,
        "costo_pedido_anual": costo_pedido_anual,
        "costo_mant_anual": costo_mant_anual,
        "costo_compra_anual": costo_compra_anual,
        "costo_total_anual": costo_total_anual
    }


#(Pregunta 1 y 2) Caso base

base = politicas_inventario(D, S, H, C, L_dias, dias_operacion)

print("=== RESULTADOS CASO BASE (H=10) ===")
for k, v in base.items():
    print(f"{k}: {v}")


#Pregunta 3. Sensibilidad: H sube a 15

H_nuevo = 15
sens = politicas_inventario(D, S, H_nuevo, C, L_dias, dias_operacion)

print("\n=== RESULTADOS CON H=15 ===")
for k, v in sens.items():
    print(f"{k}: {v}")


# Comparación directa (base vs H=15)

df = pd.DataFrame([base, sens], index=["Base_H10", "Nuevo_H15"])

cols_clave = [
    "EOQ_Q", "ROP_R", "pedidos_por_anio", "tiempo_entre_pedidos_dias",
    "costo_pedido_anual", "costo_mant_anual", "costo_total_anual"
]

print("\n=== COMPARACIÓN ===")
print(df[cols_clave].to_string())

=== RESULTADOS CASO BASE (H=10) ===
D: 5000
S: 200
H: 10
C: 300
L_dias: 5
dias_operacion: 250
EOQ_Q: 447.21359549995793
ROP_R: 100.0
pedidos_por_anio: 11.180339887498949
tiempo_entre_pedidos_dias: 22.360679774997894
costo_pedido_anual: 2236.06797749979
costo_mant_anual: 2236.06797749979
costo_compra_anual: 1500000
costo_total_anual: 1504472.1359549996

=== RESULTADOS CON H=15 ===
D: 5000
S: 200
H: 15
C: 300
L_dias: 5
dias_operacion: 250
EOQ_Q: 365.14837167011075
ROP_R: 100.0
pedidos_por_anio: 13.693063937629153
tiempo_entre_pedidos_dias: 18.257418583505537
costo_pedido_anual: 2738.6127875258308
costo_mant_anual: 2738.6127875258308
costo_compra_anual: 1500000
costo_total_anual: 1505477.2255750517

=== COMPARACIÓN ===
                EOQ_Q  ROP_R  pedidos_por_anio  tiempo_entre_pedidos_dias  costo_pedido_anual  costo_mant_anual  costo_total_anual
Base_H10   447.213595  100.0         11.180340                  22.360680         2236.067977       2236.067977       1.504472e+06
Nuevo_H15  3